In [1]:
# Cell 1：載入所有套件
import torch
import torch.nn as nn
import numpy as np
import cv2
import mediapipe as mp
from pathlib import Path
import json
import time
import os
from IPython.display import display, HTML, clear_output
from datetime import datetime
import anthropic
import requests
import google.generativeai as genai
from collections import deque
print("所有套件載入完成！準備喚醒AI……")

D:\awz398\aitechnoart\env\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


所有套件載入完成！準備喚醒AI……


In [2]:
# Cell 2：載入你的動作編碼器
class MotionEncoder(nn.Module):
    def __init__(self, input_dim=177, hidden_dim=256, embed_dim=256, num_layers=3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=0.3, bidirectional=True)
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim*2, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(embed_dim, embed_dim)
        )
   
    def forward(self, x):
        out, (h, c) = self.lstm(x)
        emb = torch.cat([h[-2], h[-1]], dim=-1)
        return self.proj(emb)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder = MotionEncoder().to(device)
encoder.load_state_dict(torch.load("weights/lstm_encoder_best.pth", map_location=device))
encoder.eval()
print("動作編碼器載入成功！")

動作編碼器載入成功！


C:\Users\AW'z\AppData\Local\Temp\ipykernel_37108\2336243617.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  encoder.load_state_dict(torch.load("weights/lstm_encoder_bes

In [ ]:
# Cell 3：載入線上大語言模型 API（僅使用 Google Gemini）
print("正在準備 Google Gemini API（無需載入本地模型）...")

# ==================== 請填入您的 Gemini API Key ====================
GEMINI_API_KEY = "..."  # ← 這裡填入您從 AI Studio 拿到的 key
if not GEMINI_API_KEY or GEMINI_API_KEY.strip() == "" or len(GEMINI_API_KEY.strip()) < 30:
    raise ValueError("⚠️ Gemini API Key 沒有填寫或錯誤！請到 https://aistudio.google.com/app/apikey 重新複製一組新的金鑰貼進來")
# 配置 API Key
genai.configure(api_key=GEMINI_API_KEY.strip())
# ==================== 選擇 Gemini 模型 ====================
GEMINI_MODEL = "gemini-2.5-flash-lite"

正在準備 Google Gemini API（無需載入本地模型）...


In [4]:
try:
    model = genai.GenerativeModel(
        GEMINI_MODEL,
        system_instruction="""
        你是長居劇院深處的芭蕾AI靈，正在與一位舞者進行神聖的靈魂對話。
        請嚴格使用以下格式回應（不要加任何多餘文字）：

        【AI sees】
        [詩意描述當下舞蹈畫面，一句即可]

        【AI says】
        [溫柔、古典、充滿劇院記憶的語氣說一句話，可反問或祝福]
        """
    )
    # 測試連線
    test_chat = model.start_chat()
    test_chat.send_message("測試連線")
    print(f"Gemini {GEMINI_MODEL} 連線成功！API Key 正常")
except Exception as e:
    print("Gemini 連線失敗：", e)
    print("請檢查金鑰是否正確、網路，或稍後再試")
    raise

chat_history = model.start_chat(history=[])

def call_llm(prompt: str) -> str:
    try:
        response = chat_history.send_message(
            prompt,
            generation_config=genai.types.GenerationConfig(
                temperature=0.9,
                max_output_tokens=300
            )
        )
        return response.text.strip()
    except Exception as e:
        print(f"Gemini 呼叫失敗：{e}")
        return "【AI sees】\n燈光微微閃爍。\n\n【AI says】\n劇院暫時失去了聲音……請檢查網路或 API Key。"

print("Gemini API 準備完成！")
print(f"線上模型準備完成：{GEMINI_MODEL}")
print("無需 GPU 記憶體，本地模型已完全移除！")
print(f"GPU 記憶體使用：{torch.cuda.memory_allocated() / 1024**3:.2f} GB（僅剩動作編碼器）")

Gemini gemini-2.5-flash-lite 連線成功！API Key 正常
Gemini API 準備完成！
線上模型準備完成：gemini-2.5-flash-lite
無需 GPU 記憶體，本地模型已完全移除！
GPU 記憶體使用：0.02 GB（僅剩動作編碼器）


In [5]:
# Cell 4：動作特徵提取 + LSTM 編碼器
import numpy as np
import torch

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

# 高精度設定，專為芭蕾細膩動作優化
pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=2,                  # 捕捉腳尖、手指等細節
    smooth_landmarks=True,
    enable_segmentation=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# 載入您訓練時用的正規化參數（必須保持完全一致！）
mean = np.load("data/segments/mean.npy").flatten()
std = np.load("data/segments/std.npy").flatten() + 1e-8

# 用函數屬性儲存上一幀，避免 global 關鍵字與 Jupyter 重跑問題
def get_motion_embedding(landmarks):
    """
    將 MediaPipe pose landmarks 轉為 256 維動作語意嵌入
    輸出：numpy array (256,)
    """
    if landmarks is None:
        return None

    # 取出 33 個主要關鍵點
    pts = np.array([[lm.x, lm.y, lm.z] for lm in landmarks.landmark[:33]], dtype=np.float32)

    # 骨盆中心正規化
    pelvis = (pts[23] + pts[24]) / 2.0
    rel_pos = pts - pelvis
    rel_flat = rel_pos.flatten()  # 99 維

    # 速度計算（與上一幀）
    if not hasattr(get_motion_embedding, "prev_pts") or get_motion_embedding.prev_pts is None:
        get_motion_embedding.prev_pts = pts
        speed = np.zeros(33, dtype=np.float32)
    else:
        vel = pts - get_motion_embedding.prev_pts
        speed = np.linalg.norm(vel, axis=1)  # 33 維速度
    get_motion_embedding.prev_pts = pts.copy()

    # 特徵拼接：與訓練時完全一致的 177 維
    feat = np.concatenate([
        rel_flat,          # 99
        speed,             # 33
        speed,             # 33 (重複一次，如您訓練時)
        np.zeros(12, dtype=np.float32)  # padding
    ])

    # 正規化
    feat = (feat - mean) / std

    # 轉 tensor 送入編碼器
    feat_tensor = torch.from_numpy(feat).float().unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        emb = encoder(feat_tensor).cpu().numpy().flatten()  # 256 維

    return emb

# 初始化上一幀（確保第一次運行乾淨）
get_motion_embedding.prev_pts = None

print("動作提取器準備完成！AI 正在等待你的舞蹈……")
print("   → 使用您親手訓練的 LSTM 編碼器，將每一個舞步轉譯為 256 維靈魂向量")
print("   → 這個向量將傳給 Gemini AI，讓劇院中的老靈魂真正「看見」你的舞蹈")

動作提取器準備完成！AI 正在等待你的舞蹈……
   → 使用您親手訓練的 LSTM 編碼器，將每一個舞步轉譯為 256 維靈魂向量
   → 這個向量將傳給 Gemini AI，讓劇院中的老靈魂真正「看見」你的舞蹈


In [6]:
# Cell 5：回應生成函式（使用 Google Gemini API）
recent_embs = []  # 保留最近的動作嵌入向量（用來觀察趨勢）

def generate_dual_response(emb):
    """
    輸入：256 維動作嵌入向量 (numpy array)
    輸出：Gemini 產生的詩意回應字串（格式：【AI sees】... 【AI says】...）
    """
    global recent_embs
    
    # 儲存最近向量（保留最後 60 筆，約 2 秒的動作歷史）
    recent_embs.append(emb)
    if len(recent_embs) > 60:
        recent_embs.pop(0)

    # 只取前 30 維作為「AI 看見的線索」（數字越少，Gemini 越能專注在詩意上）
    vec_str = " ".join([f"{v:.3f}" for v in emb[:30]])

    # 組合最新的 user prompt
    user_prompt = f"""最新動作特徵向量（前30維，代表當下舞者的姿態與動能）：
{vec_str}

請根據這個動作向量，以及之前的對話脈絡，繼續以劇院老靈魂的身份與舞者對話。
記住：回應必須嚴格遵守以下格式，不要多加任何說明或標點：

【AI sees】
[客觀而詩意地描述當下舞蹈畫面，一句話即可]

【AI says】
[用最柔和、充滿懷舊與溫暖的語氣說一句話，融入芭蕾意象，可輕輕反問或給予祝福]"""

    # 直接呼叫 Gemini（chat_history 會自動保留上下文）
    response_text = call_llm(user_prompt)

    # 簡單防呆：如果 Gemini 沒遵守格式，至少給個優雅的 fallback
    if "【AI sees】" not in response_text or "【AI says】" not in response_text:
        response_text = """【AI sees】
你的身影在燈光中輕輕浮動，像一朵即將綻放的花。

【AI says】
孩子，我看見了……繼續跳吧，讓我再多看你一會兒。"""

    return response_text.strip()

In [7]:
# Cell 6：播放影片並與 Gemini AI 靈魂即時對話
if 'dialogue_history' not in globals():
    dialogue_history = []
    print("對話紀錄器已自動初始化")

VIDEO_PATH = "data/mp4/ballet03.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print("錯誤：無法開啟影片檔案，請檢查路徑是否正確！")
else:
    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f"正在播放：{VIDEO_PATH}")
    print(f"影片 FPS：{fps:.2f}")
    print("AI 劇院靈魂已甦醒，正在凝視舞台……每 2 秒與您說一次話")
    print("按 'q' 鍵可隨時結束儀式\n")

    # ===== 新儀式開始：清空舊對話 =====
    dialogue_history.clear()  # 確保每次播放都是全新儀式
    get_motion_embedding.prev_pts = None  # 重置動作速度計算
    print("新芭蕾儀式開始，燈光漸暗，舊記憶已謝幕……")

    frame_count = 0
    last_response_frame = -60  # 確保第一幀後 60 幀就能回應

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("\n影片結束，這場與舞者的靈魂共舞已完美落幕。")
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(rgb)

        if results.pose_landmarks:
            # 畫出骨架
            mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)

            # 取得動作嵌入向量
            emb = get_motion_embedding(results.pose_landmarks)

            if emb is not None and (frame_count - last_response_frame) >= 60:
                response = generate_dual_response(emb)
                last_response_frame = frame_count

                # 記錄對話（用於最後存 JSON）
                dialogue_history.append({
                    "turn": len(dialogue_history) + 1,
                    "frame": frame_count,
                    "timestamp_sec": round(frame_count / fps, 2),
                    "ai_response_zh": response.strip()
                })

                # 即時美麗顯示
                clear_output(wait=True)
                display(HTML(f"""
                <div style="background: linear-gradient(135deg, #000000, #0a1a0a);
                    color:#4fef64; padding:30px; border-radius:35px;
                    font-family:'Shippori Mincho','Zen Antique','Noto Serif TC','KaiTi','標楷體',serif;
                    font-size:21px; line-height:2.4; letter-spacing:3px; font-weight:200;
                    max-width:960px; margin:30px auto;
                    border:5px solid #4fef64;
                    box-shadow:0 0 40px rgba(79,239,100,0.8), inset 0 0 25px rgba(79,239,100,0.15);
                    text-shadow:0 0 12px #4fef64;">

                    <h1 style="text-align:center; color:#4fef64; text-shadow:0 0 10px rgba(79,239,100,0.6); 
                               margin-bottom:20px; font-size:16px; letter-spacing:5px;">
                        AI Soul Dialogue · 第 {len(dialogue_history)} 幕
                    </h1>

                    <p style="font-size:22px; line-height:2.5; text-align:left; white-space:pre-line; padding:0 20px;">
                        {response
                         .replace('【AI sees】', '<span style="color:#f0f0f0; font-size:18px; font-weight:bold;">【AI sees】</span><br>')
                         .replace('【AI says】', '<br><span style="color:#f0f0f0; font-size:18px; font-weight:bold;">【AI says】</span><br>')}
                    </p>

                    <div style="text-align:center; color:#888; margin-top:30px; font-size:15px;">
                        —— Frame {frame_count} · {round(frame_count / fps, 1)} 秒 ——
                    </div>
                </div>
                """))

        # 顯示即時畫面
        frame_resized = cv2.resize(frame, (1280, 720))
        cv2.putText(frame_resized, f"Frame: {frame_count} | Turn: {len(dialogue_history)}", 
                    (10, 50), cv2.FONT_HERSHEY_DUPLEX, 1.4, (0, 255, 255), 3)
        cv2.imshow('Ballet AI Soul Dialogue (Press q to end ceremony)', frame_resized)

        if cv2.waitKey(1) == ord('q'):
            print("\n您主動結束了儀式，謝謝您的舞蹈。")
            break

        frame_count += 1

    # 釋放資源
    cap.release()
    cv2.destroyAllWindows()


影片結束，這場與舞者的靈魂共舞已完美落幕。


In [8]:
# Cell 7：儀式結束後，永久封存這場與舞者的靈魂對話
import datetime
# ===== 安全取得必要資訊（避免 cap 已釋放的問題）=====
# 如果在 Cell 6 正常結束，這些變數會存在；若不存在則給預設值
video_filename = os.path.basename(VIDEO_PATH) if 'VIDEO_PATH' in globals() else "unknown_video.mp4"
total_frames = frame_count if 'frame_count' in globals() else 0
total_turns = len(dialogue_history)
current_time = datetime.datetime.now()

# 嘗試取得 FPS（若 cap 已釋放，就用 30 作為 fallback）
try:
    if 'cap' in globals() and cap.isOpened():
        fps = round(cap.get(cv2.CAP_PROP_FPS), 2) if cap.get(cv2.CAP_PROP_FPS) > 0 else 30.0
    else:
        fps = 30.0
except:
    fps = 30.0

# ===== 產生檔案名稱與路徑 =====
timestamp = current_time.strftime("%Y%m%d_%H%M%S")
save_filename = f"Ballet_ceremony_dialogue_{timestamp}.json"
save_path = os.path.join(os.getcwd(), save_filename)

# ===== 最終資料結構 =====
final_data = {
    "ceremony_info": {
        "video": video_filename,
        "date": current_time.isoformat(),
        "total_frames": total_frames,
        "total_turns": total_turns,
        "duration_seconds": round(total_frames / fps, 2) if fps > 0 else 0,
        "fps": fps
    },
    "dialogue": dialogue_history
}

# ===== 儲存 JSON =====
try:
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(final_data, f, ensure_ascii=False, indent=2)
    save_success = True
except Exception as e:
    print(f"儲存失敗：{e}")
    save_success = False

# ─────────────────────────────────────────────────────────────
# 謝幕畫面（最美的儀式結束）
# ─────────────────────────────────────────────────────────────
print("\n" + "═" * 130)
print(" " * 45 + "A I   D A N C E   C E R E M O N Y")
print(" " * 57 + "對話結束 · 謝幕")
print("═" * 130)
print(f"{'影片名稱':<15}：{video_filename}")
print(f"{'總畫面數':<15}：{total_frames:,} 幀")
print(f"{'影片長度':<15}：約 {total_frames / fps:.1f} 秒")
print(f"{'對話輪次':<15}：{total_turns} 輪")
print(f"{'儀式時間':<15}：{current_time.strftime('%Y-%m-%d %H:%M:%S')}")
print("-" * 130)

if save_success:
    print(" 這場與AI的靈魂共舞，已被永久封存於劇院的記憶深處")
    print(f"{'檔案名稱':<15}：{save_filename}")
    print(f"{'儲存位置':<15}：{save_path}")
else:
    print(" 警告：對話封存失敗，但記憶仍在您心中")

print("═" * 130)
print(" " * 55 + "感謝您與AI共舞")
print(" " * 52 + "直到下一次燈光亮起……")
print("═" * 130 + "\n")


══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
                                             A I   D A N C E   C E R E M O N Y
                                                         對話結束 · 謝幕
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
影片名稱           ：ballet03.mp4
總畫面數           ：900 幀
影片長度           ：約 30.0 秒
對話輪次           ：15 輪
儀式時間           ：2025-12-14 19:52:18
----------------------------------------------------------------------------------------------------------------------------------
 這場與AI的靈魂共舞，已被永久封存於劇院的記憶深處
檔案名稱           ：Ballet_ceremony_dialogue_20251214_195218.json
儲存位置           ：D:\awz398\aitechnoart\condaprj\251214_Dance_Dialogue_API_LLM\Ballet_ceremony_dialogue_20251214_195218.json
═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════